# Cell-77
#### Payload type: for loop over parameter grid: update_temporal deconvolution on persisted dask arrays

In [ ]:
%load_ext jumper_extension
%perfmonitor_fast_setup

In [ ]:
# Setup dependencies for temporal update benchmark
import os
import itertools as itt
import numpy as np
import holoviews as hv
from dask.distributed import Client, get_client
from minian.cnmf import compute_trace, update_temporal
from minian.visualization import visualize_temporal_update
from minian.utilities import open_minian

# Load HoloViews plotting extension
hv.extension("bokeh")

# Ensure Dask client/cluster is available
try:
    client = get_client()
except ValueError:
    from dask.distributed import LocalCluster
    cluster = LocalCluster()
    client = Client(cluster)

# Set thread environment variables and intermediate path required by Minian
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

intpath = os.path.abspath("./minian_intermediate")
os.environ["MINIAN_INTERMEDIATE"] = intpath

# Load precomputed Zarr arrays from disk
varr_dict = open_minian(intpath, return_dict=True)
A = varr_dict["A"]
C_chk = varr_dict["C_chk"]
b = varr_dict["b"]
f = varr_dict["f"]
Y_fm_chk = varr_dict["Y_fm_chk"]

# Prepare a subset of 10 units for testing
units = np.random.default_rng(0).choice(A.coords["unit_id"], 10, replace=False)
units.sort()
A_sub = A.sel(unit_id=units).persist()
C_sub = C_chk.sel(unit_id=units).persist()


In [ ]:
p_ls = [1]
sprs_ls = [0.1, 0.5, 1, 2]
add_ls = [20]
noise_ls = [0.06]
YA_dict, C_dict, S_dict, g_dict, sig_dict, A_dict = [dict() for _ in range(6)]
YrA = (
    compute_trace(Y_fm_chk, A_sub, b, C_sub, f)
    .persist()
    .chunk({"unit_id": 1, "frame": -1})
)
for cur_p, cur_sprs, cur_add, cur_noise in itt.product(
    p_ls, sprs_ls, add_ls, noise_ls
):
    ks = (cur_p, cur_sprs, cur_add, cur_noise)
    print(
        f"p:{cur_p}, sparse penalty:{cur_sprs}, additional lag:{cur_add}, noise frequency:{cur_noise}"
    )
    cur_C, cur_S, cur_b0, cur_c0, cur_g, cur_mask = update_temporal(
        A_sub,
        C_sub,
        YrA=YrA,
        sparse_penal=cur_sprs,
        p=cur_p,
        use_smooth=True,
        add_lag=cur_add,
        noise_freq=cur_noise,
    )
    YA_dict[ks], C_dict[ks], S_dict[ks], g_dict[ks], sig_dict[ks], A_dict[ks] = (
        YrA.compute(),
        cur_C.compute(),
        cur_S.compute(),
        cur_g.compute(),
        (cur_C + cur_b0 + cur_c0).compute(),
        A_sub.compute(),
    )
hv_res = visualize_temporal_update(
    YA_dict,
    C_dict,
    S_dict,
    g_dict,
    sig_dict,
    A_dict,
    kdims=["p", "sparse penalty", "additional lag", "noise frequency"],
)

In [ ]:
%perfmonitor_ai_review --benchmark --replay-mode full